# CISB5123 Text Analytics
## Lab Assignment 3 — Topic modeling
**Name:** OSAMA MOHAMMED ALI ABDASLAM  
**Student ID:** SW01084153

---

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

import gensim
from gensim import corpora
from gensim.models import CoherenceModel

import re

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\osaal\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\osaal\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## Load Dataset (Only 'text' Column)

In [2]:
df = pd.read_csv('news_dataset.csv')

# Keep only 'text' column
df = df[['text']]

# Remove null values
df.dropna(inplace=True)

df.head()

,text
0,I was wondering if anyone out there could enli...
1,I recently posted an article asking what kind ...
2,\nIt depends on your priorities. A lot of peo...
3,an excellent automatic can be found in the sub...
4,: Ford and his automobile. I need information...


## Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    # Lowercase
    text = text.lower()
    
    # Remove punctuation & numbers
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Tokenization
    tokens = text.split()
    
    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    
    # Stemming
    tokens = [stemmer.stem(word) for word in tokens]
    
    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return tokens

processed_text = df['text'].apply(preprocess)

## Create Dictionary & Corpus

In [ ]:
dictionary = corpora.Dictionary(processed_text)

# Filter extremes to remove very rare and very common words
dictionary.filter_extremes(no_below=5, no_above=0.5)

corpus = [dictionary.doc2bow(text) for text in processed_text]

## Perform LDA (4 Topics)

In [5]:
lda_model = gensim.models.LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=4,
    random_state=42,
    passes=10,
    alpha='auto'
)

## Evaluate with Coherence Score

In [6]:
coherence_model = CoherenceModel(
    model=lda_model,
    texts=processed_text,
    dictionary=dictionary,
    coherence='c_v'
)

coherence_score = coherence_model.get_coherence()

print("Coherence Score:", coherence_score)

Coherence Score: 0.5490443813463459


## Display Topics

In [10]:
topics = lda_model.print_topics(num_words=10)

for i, topic in enumerate(topics):
    print(f"\nTopic {i + 1}: {topic}")


Topic 1: (0, '0.015*"maxaxaxaxaxaxaxaxaxaxaxaxaxaxax" + 0.014*"use" + 0.008*"get" + 0.008*"window" + 0.007*"one" + 0.007*"like" + 0.006*"would" + 0.006*"work" + 0.006*"problem" + 0.006*"know"')

Topic 2: (1, '0.021*"x" + 0.015*"use" + 0.015*"key" + 0.010*"encrypt" + 0.008*"b" + 0.008*"file" + 0.007*"system" + 0.007*"db" + 0.007*"program" + 0.007*"chip"')

Topic 3: (2, '0.009*"year" + 0.009*"game" + 0.007*"team" + 0.006*"play" + 0.005*"new" + 0.004*"player" + 0.004*"first" + 0.004*"last" + 0.004*"armenian" + 0.003*"win"')

Topic 4: (3, '0.009*"would" + 0.009*"peopl" + 0.009*"one" + 0.006*"think" + 0.006*"dont" + 0.006*"say" + 0.006*"know" + 0.005*"like" + 0.005*"go" + 0.005*"u"')


The LDA model produced a coherence score of 0.549, which indicates a moderate level of topic quality. This suggests that the generated topics are somewhat interpretable but not highly distinct.

From the extracted topics, some meaningful themes can be identified. For example, Topic 2 is related to technology and encryption, while Topic 3 clearly represents sports-related content. However, other topics contain generic or less informative words, reducing their interpretability.

Additionally, the presence of noisy and irrelevant tokens indicates that the preprocessing step can be improved. Overall, while the model demonstrates the ability to extract underlying topics from unlabeled data, further tuning and better preprocessing could enhance the coherence and clarity of the topics.